# NL Team Trends — Exploratory Data Analysis

This notebook provides starter EDA for the National League team trends dataset.

## Overview

Loads the key data files and generates initial visualizations to explore:
- Franchise win-loss trends over time
- Era-based dominance patterns
- Head-to-head rivalry comparisons
- Championship distribution

## Setup

```bash
pip install -r ../requirements.txt
```

## Usage

Open in Jupyter:
```bash
jupyter notebook notebooks/01_eda_nl_standings.ipynb
```

Or run with pipenv:
```bash
pipenv run jupyter notebook notebooks/01_eda_nl_standings.ipynb
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Style settings
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data")

# --------------------------------------------------------------
# 1. Load All-Time Franchise Records
# --------------------------------------------------------------
print("Loading NL franchise records...")
records = pd.read_csv(os.path.join(DATA_DIR, "nl_all_time_records.csv"))
records = records.sort_values("ws_titles", ascending=False)

In [ ]:
print(f"\nLoaded {len(records)} NL franchises")
records[["franchise", "ws_titles", "total_wins", "total_losses", "win_pct"]].head(15)

In [ ]:
# --------------------------------------------------------------
# 2. WS Title Distribution
# --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette("Blues_d", len(records))
ax.barh(records["franchise"], records["ws_titles"], color=colors)
ax.set_xlabel("World Series Titles")
ax.set_title("NL Franchises by World Series Titles (through 2025)")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "..", "charts", "ws_titles_by_team.png"), dpi=150)
plt.show()

In [ ]:
# --------------------------------------------------------------
# 3. Win % vs Win-Differential (bubble = total games)
# --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    records["total_wins"] - records["total_losses"],
    records["win_pct"],
    s=records["total_games"] / 100,
    c=records["ws_titles"],
    cmap="viridis",
    alpha=0.7,
    edgecolors="black",
    linewidth=0.5,
)
for _, row in records.iterrows():
    ax.annotate(
        row["franchise"],
        (row["total_wins"] - row["total_losses"], row["win_pct"]),
        fontsize=8,
        ha="left",
        xytext=(5, 0),
        textcoords="offset points",
    )
ax.set_xlabel("Win Differential (W - L)")
ax.set_ylabel("Win Percentage")
ax.set_title("NL Franchise Win% vs Win Differential (bubble = total games)")
plt.colorbar(scatter, label="WS Titles")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "..", "charts", "win_pct_vs_differential.png"), dpi=150)
plt.show()

In [ ]:
# --------------------------------------------------------------
# 4. WS Titles by Era
# --------------------------------------------------------------
print("\n\nLoading NL pennant winners...")
pennants = pd.read_csv(os.path.join(DATA_DIR, "nl_pennant_winners.csv"))
ws_won = pennants[pennants["ws_result"] == "Won"]

era_bins = [1876, 1900, 1920, 1940, 1960, 1980, 2026]
era_labels = [
    "Founding\n1876-1900",
    "Dead Ball\n1901-1920",
    "Pre-War\n1921-1940",
    "Cards Dynasty\n1941-1960",
    "Expansion\n1961-1980",
    "Braves Era\n1981-2000",
    "Resurgence\n2001-2025",
]

ws_by_era = {}
for i in range(len(era_bins) - 1):
    mask = (
        pd.to_numeric(ws_won["ws_yr"], errors="coerce") >= era_bins[i]
    ) & (
        pd.to_numeric(ws_won["ws_yr"], errors="coerce") < era_bins[i + 1]
    )
    ws_by_era[era_labels[i]] = mask.sum()

fig, ax = plt.subplots(figsize=(14, 5))
era_df = pd.Series(ws_by_era).sort_values()
ax.barh(era_df.index, era_df.values, color=sns.color_palette("Reds_r", len(era_df)))
ax.set_xlabel("World Series Titles")
ax.set_title("NL WS Titles by Era")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "..", "charts", "ws_titles_by_era.png"), dpi=150)
plt.show()

In [ ]:
# --------------------------------------------------------------
# 5. H2Rivalry Comparison Chart
# --------------------------------------------------------------
print("\n\nLoading H2H rivalry data...")
h2h = pd.read_csv(os.path.join(DATA_DIR, "nl_h2h_rivalries_detailed.csv"))

top_rivalries = h2h.nlargest(10, "total_games")
fig, ax = plt.subplots(figsize=(12, 6))
bar_width = 0.35
x = range(len(top_rivalries))
ax.bar([i - bar_width/2 for i in x], top_rivalries["team_1_wins"], bar_width, label=top_rivalries["team_1"], color="#005A9C")
ax.bar([i + bar_width/2 for i in x], top_rivalries["team_2_wins"], bar_width, label=top_rivalries["team_2"], color="#BA0C2F")
ax.set_xticks(x)
ax.set_xticklabels([f"{r['team_1']}\nvs\n{r['team_2']}" for _, r in top_rivalries.iterrows()], fontsize=8)
ax.set_ylabel("Total Wins")
ax.set_title("Top 10 NL Rivalries by Total Games Played")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "..", "charts", "top_h2h_rivalries.png"), dpi=150)
plt.show()

print("\nEDA complete! Check the charts/ directory for output images.")